# Plasma property test fixtures explorer

This notebook rebuilds the fixtures from `conftest.py` and shows what each object looks like with the **`atomic_dataset`** from the plasma property tests.

- **Fixtures:** `tardis/plasma/properties/tests/conftest.py`
- **Abundances:** H = 0.8, He = 0.2 in every shell (from `grid/tests/data/example.yml`)
- **Atom data:** `kurucz_cd23_chianti_H_He_latest.h5` (same file as pytest `atomic_dataset`)

If loading fails, set the environment variable `TARDIS_REGRESSION_DATA` to your tardisbase regression data directory.

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from tardis.io.atom_data.base import AtomData
from tardis.plasma.properties.general import SelectedAtoms

NUMBER_OF_CELLS = 20


def find_atomic_data_fname() -> Path:
    """Same atom data file as pytest fixture atomic_data_fname."""
    fname = "kurucz_cd23_chianti_H_He_latest.h5"
    candidates = []

    env = os.environ.get("TARDIS_REGRESSION_DATA")
    if env:
        candidates.append(Path(os.path.expandvars(env)).expanduser() / "atom_data" / fname)

    repo_root = Path.cwd()
    for base in [
        repo_root / "../../../../tardis-regression-data",
        repo_root / "../../../tardis-regression-data",
        Path.home() / "tardisDev/tardis-regression-data",
        repo_root / "../../../../tardisbase/tardisbase/testing/regression_data",
        repo_root / "../../../tardisbase/tardisbase/testing/regression_data",
        Path.home() / "tardisbase/tardisbase/testing/regression_data",
        Path.home() / "tardisDev/tardisbase/tardisbase/testing/regression_data",
    ]:
        candidates.append(base.resolve() / "atom_data" / fname)

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find kurucz_cd23_chianti_H_He_latest.h5. "
        "Set TARDIS_REGRESSION_DATA to your tardisbase regression_data directory."
    )


atomic_data_fname = find_atomic_data_fname()
atomic_dataset = AtomData.from_hdf(atomic_data_fname)

print(f"Loaded atom data: {atomic_data_fname}")
print(f"MD5: {atomic_dataset.md5}")

Loaded atom data: /home/rishmitar/tardisDev/tardis-regression-data/atom_data/kurucz_cd23_chianti_H_He_latest.h5
MD5: 5d80fa4ae0638469bf1ff281b6ca2a94


In [3]:
# --- conftest fixtures (same logic as conftest.py) ---

abundance = pd.DataFrame(
    data=[[0.8] * NUMBER_OF_CELLS, [0.2] * NUMBER_OF_CELLS],
    index=pd.Index([1, 2], name="atomic_number"),
    columns=range(NUMBER_OF_CELLS),
    dtype=np.float64,
)

density = np.ones(NUMBER_OF_CELLS) * 1e-14

atomic_mass = atomic_dataset.atom_data.reindex(abundance.index)["mass"]

number_density = abundance.mul(density, axis=1).div(atomic_mass, axis=0)

selected_atoms = SelectedAtoms(None).calculate(number_density)

print("Types:")
print(f"  abundance:       {type(abundance).__name__}  shape={abundance.shape}")
print(f"  density:         {type(density).__name__}  shape={density.shape}")
print(f"  atomic_mass:     {type(atomic_mass).__name__}  len={len(atomic_mass)}")
print(f"  number_density:  {type(number_density).__name__}  shape={number_density.shape}")
print(f"  selected_atoms:  {type(selected_atoms).__name__}  values={list(selected_atoms)}")

Types:
  abundance:       DataFrame  shape=(2, 20)
  density:         ndarray  shape=(20,)
  atomic_mass:     Series  len=2
  number_density:  DataFrame  shape=(2, 20)
  selected_atoms:  Index  values=[1, 2]


## `abundance` — mass fractions per element and shell

Rows = atomic numbers (1 = H, 2 = He). Columns = shell index (0–19). Values are **dimensionless** mass fractions (same in every shell here).

In [4]:
display(abundance)
print("First 3 shells:")
display(abundance.iloc[:, :3])

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
atomic_number,,,,,,,,,,,,,,,,,,,,
1,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8
2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2,0.2


First 3 shells:


,0,1,2
atomic_number,,,
1,0.8,0.8,0.8
2,0.2,0.2,0.2


## `density` — mass density per shell

1D array, one value per shell (g/cm³). Same density in every shell in this fixture.

In [5]:
pd.Series(density, name="density [g/cm^3]", index=range(NUMBER_OF_CELLS))

0     1.000000e-14
1     1.000000e-14
2     1.000000e-14
3     1.000000e-14
4     1.000000e-14
5     1.000000e-14
6     1.000000e-14
7     1.000000e-14
8     1.000000e-14
9     1.000000e-14
10    1.000000e-14
11    1.000000e-14
12    1.000000e-14
13    1.000000e-14
14    1.000000e-14
15    1.000000e-14
16    1.000000e-14
17    1.000000e-14
18    1.000000e-14
19    1.000000e-14
Name: density [g/cm^3], dtype: float64

## `atomic_mass` — from `atomic_dataset.atom_data`

Series indexed by atomic number (g). Used to convert mass fraction × density → elemental number density.

In [6]:
display(atomic_mass.to_frame(name="mass [g]"))
display(atomic_dataset.atom_data.loc[abundance.index])  # full atom_data rows for H and He

,mass [g]
atomic_number,
1,1.673782e-24
2,6.646476e-24


N.,symbol,name,mass
atomic_number,,,
1,H,Hydrogen,1.673782e-24
2,He,Helium,6.646476e-24


## `number_density` — (abundance × density) / atomic_mass

DataFrame: rows = elements, columns = shells. Units: number density (cm⁻³) per element per zone.

Formula matches iip `NumberDensity.calculate(atomic_mass, abundance, density)`.

In [7]:
display(number_density)
print("\nShell 0 only:")
display(number_density[[0]])
print("\nSame value in every shell (uniform abundance & density):")
display(number_density[0].to_frame(name="N_i shell 0 [cm^-3]"))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
atomic_number,,,,,,,,,,,,,,,,,,,,
1,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09,4.779596e+09
2,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08,3.009113e+08



Shell 0 only:


,0
atomic_number,
1,4.779596e+09
2,3.009113e+08



Same value in every shell (uniform abundance & density):


,N_i shell 0 [cm^-3]
atomic_number,
1,4.779596e+09
2,3.009113e+08


## `selected_atoms` — from `SelectedAtoms.calculate(number_density)`

In `tardis.plasma.properties.general`, this is just `number_density.index` (atomic numbers present in the number-density table).

In [8]:
selected_atoms

Index([1, 2], dtype='int64', name='atomic_number')

## How the pieces connect

```
abundance (2 × 20)     density (20,)
        \                   /
         ×  (broadcast per shell)
                |
         ÷ atomic_mass (per row, axis=0)
                |
      number_density (2 × 20)
                |
         .index → selected_atoms [1, 2]
                |
    atomic_dataset.levels/lines filtered by selected_atoms in tests
```